In [1]:
import os
import requests
import pandas as pd
from datetime import datetime, UTC
from google.cloud import bigquery

In [2]:
PROJECT_ID = os.getenv("PROJECT_ID", "chrome-epigram-472216-t5")
DATASET_ID = os.getenv("DATASET_ID", "raw")
TABLE_ID = os.getenv("TABLE_ID", "fx_rates")
API_URL = os.getenv("API_URL", "https://open.er-api.com/v6/latest/USD")

FULL_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

In [5]:
def extract_fx_rates(api_url: str) -> dict:
    response = requests.get(api_url, timeout=30)
    response.raise_for_status()
    return response.json()

In [7]:
def transform_fx_rates(data: dict) -> pd.DataFrame:
    extracted_at = datetime.now(UTC)

    rows = [
        {
            "base_currency": data["base_code"],
            "target_currency": currency,
            "rate": float(rate),
            "extracted_at": extracted_at,
            "extraction_date": extracted_at.date(),
        }
        for currency, rate in data["rates"].items()
    ]

    return pd.DataFrame(rows)

In [9]:
def load_to_bigquery(df: pd.DataFrame, table_id: str) -> None:
    client = bigquery.Client(project=PROJECT_ID)

    job_config = bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField("base_currency", "STRING"),
            bigquery.SchemaField("target_currency", "STRING"),
            bigquery.SchemaField("rate", "FLOAT"),
            bigquery.SchemaField("extracted_at", "TIMESTAMP"),
            bigquery.SchemaField("extraction_date", "DATE"),
        ],
        write_disposition="WRITE_TRUNCATE",  # reemplaza la tabla completa
    )

    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=job_config,
    )

    job.result()

    print(f"Loaded {len(df)} rows into {table_id}")

In [11]:
def validate_load(table_id: str) -> pd.DataFrame:
    client = bigquery.Client(project=PROJECT_ID)

    query = f"""
    SELECT
      COUNT(*) AS total_rows,
      COUNT(DISTINCT target_currency) AS distinct_currencies,
      MIN(extracted_at) AS first_extraction,
      MAX(extracted_at) AS last_extraction
    FROM `{table_id}`
    """

    return client.query(query).to_dataframe()

In [33]:
def main():
    data = extract_fx_rates(API_URL)
    df = transform_fx_rates(data)
    print(df.head())
    print("")
    print("")
    load_to_bigquery(df, FULL_TABLE_ID)
    print("")
    print("")
    validation_df = validate_load(FULL_TABLE_ID)
    print(validation_df)


if __name__ == "__main__":
    main()

  base_currency target_currency        rate                     extracted_at  \
0           USD             USD    1.000000 2026-04-27 03:03:35.404975+00:00   
1           USD             AED    3.672500 2026-04-27 03:03:35.404975+00:00   
2           USD             AFN   63.560563 2026-04-27 03:03:35.404975+00:00   
3           USD             ALL   81.575164 2026-04-27 03:03:35.404975+00:00   
4           USD             AMD  371.969139 2026-04-27 03:03:35.404975+00:00   

  extraction_date  
0      2026-04-27  
1      2026-04-27  
2      2026-04-27  
3      2026-04-27  
4      2026-04-27  


Loaded 166 rows into chrome-epigram-472216-t5.raw.fx_rates


   total_rows  distinct_currencies                 first_extraction  \
0         166                  166 2026-04-27 03:03:35.404975+00:00   

                   last_extraction  
0 2026-04-27 03:03:35.404975+00:00  
